
# Min-K%++ Fair Comparison (standard vs attnres)

**Goal:** keep one concise pipeline for **dual-model comparison** using **Min-K%++ only**.

This notebook now centers on:
- formula-exact `Min-K%++` scoring at `k=10/20/40`,
- one shared evaluation pipeline for `standard` and `attnres`,
- focused follow-up on model-vs-model summary and cross-k stability.


In [ ]:
# =============================================================================
# Cell 1. Environment & Imports
# =============================================================================
import sys
import json
import math
from copy import deepcopy
from pathlib import Path
from collections import defaultdict

import torch
import matplotlib.pyplot as plt
import numpy as np

# --- locate repo root ---
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'toygpt2').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('repo root not found (should contain toygpt2/)')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_NAME = 'min_k_and_min_kpp_minimal'
OUTPUT_DIR = REPO_ROOT / 'output' / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def output_path(filename: str) -> Path:
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _to_serializable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_serializable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return value


def save_json_artifact(payload, filename: str) -> Path:
    path = output_path(filename)
    path.write_text(json.dumps(_to_serializable(payload), ensure_ascii=False, indent=2), encoding='utf-8')
    print('[saved]', path)
    return path


def save_dataframe_artifact(df, filename: str) -> Path:
    path = output_path(filename)
    df.to_csv(path, index=False)
    print('[saved]', path)
    return path


def save_figure_artifact(filename: str, fig=None, *, dpi: int = 180) -> Path:
    fig = plt.gcf() if fig is None else fig
    path = output_path(filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('[saved]', path)
    return path

from scripts.evaluate import load_checkpoint
from toygpt2.config import DataConfig, ModelConfig
from data.data import build_dataloaders

print('REPO_ROOT:', REPO_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Imports OK')



## Cell 2. Method Definition (Locked)

### Min-K%++ (Zhang et al., 2024)
**Paper:** *Min-K%++: Improved Baseline for Detecting Pre-Training Data from Large Language Models*  
**Repo:** [zjysteven/mink-plus-plus](https://github.com/zjysteven/mink-plus-plus)

**Definition:**
Given a sequence $X = (x_1, \ldots, x_N)$ and a target model $M$:
1. Teacher-forced forward: compute $\log p_M(v \mid x_{<t})$ and $p_M(v \mid x_{<t})$ for all vocab tokens $v$ at each position $t$.
2. Compute the probability-weighted mean of log-probabilities:
   $\mu_t = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot \log p_M(v \mid x_{<t})$.
3. Compute the probability-weighted variance:
   $\sigma_t^2 = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot [\log p_M(v \mid x_{<t})]^2 - \mu_t^2$.
4. Normalize the observed token log-probability:
   $z_t = rac{\log p_M(x_t \mid x_{<t}) - \mu_t}{\sigma_t}$.
5. Select the bottom $k\%$ of the per-token $z_t$ values and average them:
   $	ext{Min-K\%++}(X) = rac{1}{k_{	ext{len}}} \sum_{i \in 	ext{bottom-}k\% 	ext{ of } z} z_i$.

**Key:** $\mu_t$ and $\sigma_t$ are computed from the **probability-weighted** expectation
$\mathbb{E}_{V \sim p_M(\cdot|x_{<t})}[\log p_M(V|x_{<t})]$, not a uniform average over the vocabulary.
This matches the official implementation.

### Implementation status
- **Min-K%++:** formula-exact reproduction of the z-score version.
- $\mu_t$ and $\sigma_t$ are computed over the **full vocabulary** at each position using `probs * log_probs`.
- **Simplifications vs paper:** tiny model, smoke-test scale, and TinyStories member/non-member split instead of a large benchmark.
- This notebook is therefore **formula-exact, scale-limited**.


In [ ]:
# =============================================================================
# Cell 3. Load Checkpoint & Model
# =============================================================================
# Load BOTH checkpoints (standard + attnres) with robust path candidates.

MODEL_VARIANTS = ['standard', 'attnres']

CKPT_CANDIDATE_PATHS = {
    'standard': [
        REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'standard' / 'ckpt_standard_last.pt',
    ],
    'attnres': [
        REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'attnres' / 'ckpt_attnres_last.pt',
    ],
}

# Optional local fallback for standard smoke checkpoint.
CKPT_CANDIDATE_PATHS['standard'].append(
    REPO_ROOT / 'toygpt2_runs' / 'notebook_bootstrap' / 'ckpt_standard_last.pt'
)

CKPT_CANDIDATE_PATHS['attnres'].append(
    REPO_ROOT / 'toygpt2_runs' / 'notebook_bootstrap' / 'ckpt_attnres_last.pt'
)

device = torch.device('cpu')
models = {}
experiments = {}
checkpoints = {}
resolved_ckpts = {}

for variant in MODEL_VARIANTS:
    resolved = None
    tried = []
    for cand in CKPT_CANDIDATE_PATHS[variant]:
        tried.append(str(cand))
        if cand.exists():
            resolved = cand
            break

    if resolved is None:
        tried_block = ''.join(f'- {t}{chr(10)}' for t in tried)
        raise FileNotFoundError(
            'No checkpoint found for variant=' + variant + '. Tried:' + chr(10) + tried_block
        )

    model_i, exp_i, ckpt_i = load_checkpoint(resolved, device=device)
    model_i = model_i.to(device).eval()

    models[variant] = model_i
    experiments[variant] = exp_i
    checkpoints[variant] = ckpt_i
    resolved_ckpts[variant] = resolved

    mc_i = exp_i.model
    print(f'[{variant}] checkpoint: {resolved}')
    print(f'[{variant}] model_type={ckpt_i.get("model_type")} step={ckpt_i.get("step")}')
    print(f'[{variant}] n_layer={mc_i.n_layer} n_head={mc_i.n_head} n_embd={mc_i.n_embd}')
    print(f'[{variant}] vocab_size={mc_i.vocab_size} block_size={mc_i.block_size}')
    print(f'[{variant}] params={model_i.num_parameters():,}')
    print('-' * 80)

# Backward-compatible defaults for cells that expect single-model handles.
model = models['standard']
experiment = experiments['standard']
checkpoint = checkpoints['standard']
mc = experiment.model



In [ ]:

# =============================================================================
# Cell 5. Teacher-Forced Token Scoring Helper for Min-K%++
# =============================================================================
# This helper runs a teacher-forced forward pass and returns:
#   1. the log-probability of the observed next token,
#   2. the probability-weighted mean of vocab log-probabilities at each position,
#   3. the probability-weighted standard deviation at each position.

@torch.no_grad()
def compute_per_token_logprobs_and_stats(input_ids: torch.Tensor, targets: torch.Tensor):
    """
    Returns the per-token quantities required by Min-K%++.

    Args:
        input_ids: [B, L] token ids fed to the model
        targets:   [B, L] ground-truth next-token ids

    Returns:
        token_logprobs: [B, L] log p(x_t | x_{<t})
        mu:             [B, L] probability-weighted mean of log-probs over vocab
        sigma:          [B, L] probability-weighted std of log-probs over vocab
    """
    outputs = model(input_ids.to(device))
    logits = outputs['logits']

    log_probs = torch.log_softmax(logits, dim=-1)
    probs = torch.softmax(logits, dim=-1)

    token_logprobs = torch.gather(
        log_probs, dim=-1, index=targets.to(device).unsqueeze(-1)
    ).squeeze(-1)

    mu = (probs * log_probs).sum(dim=-1)
    second_moment = (probs * log_probs.pow(2)).sum(dim=-1)
    variance = (second_moment - mu.pow(2)).clamp(min=1e-10)
    sigma = variance.sqrt()

    return token_logprobs.cpu(), mu.cpu(), sigma.cpu()


In [ ]:

# =============================================================================
# Cell 6. Min-K%++ Implementation
# =============================================================================
# Exact reproduction of Zhang et al. (2024):
#   1. Compute per-token logprobs, mu, and sigma.
#   2. z_t = (logprob_t - mu_t) / sigma_t.
#   3. Sort z-scores ascending and keep the bottom k%.
#   4. Average the selected z-scores.

# mu_t = E_{V~p}[log p(V|x_{<t})] = sum_v p(v|x_{<t}) * log p(v|x_{<t})
# sigma_t = sqrt(Var_{V~p}[log p(V|x_{<t})])
# These are probability-weighted and match the official Min-K%++ repo.

def min_k_pp_score(
    token_logprobs: torch.Tensor,
    mu: torch.Tensor,
    sigma: torch.Tensor,
    k_percent: float,
) -> torch.Tensor:
    """
    Compute Min-K%++ score for a batch of sequences.

    Args:
        token_logprobs: [B, L] per-token log-probabilities
        mu:             [B, L] prob-weighted mean of log-probs at each position
        sigma:          [B, L] prob-weighted std of log-probs at each position
        k_percent:      float in (0, 100]

    Returns:
        scores: [B] Min-K%++ score for each sequence
    """
    B, L = token_logprobs.shape
    k_len = max(1, int(L * k_percent / 100.0))

    z = (token_logprobs - mu) / sigma.clamp(min=1e-8)
    sorted_z, _ = torch.sort(z, dim=-1)
    bottom_k = sorted_z[:, :k_len]
    scores = bottom_k.mean(dim=-1)
    return scores


In [ ]:
# =============================================================================
# Cell 9. AUC helper
# =============================================================================

def compute_auroc_manual(member_scores, nonmember_scores):
    """
    Manual AUC-ROC computation without sklearn.
    Convention: higher score = more likely member.
    """
    m_scores = np.asarray(member_scores)
    nm_scores = np.asarray(nonmember_scores)
    n_m = len(m_scores)
    n_nm = len(nm_scores)
    if n_m == 0 or n_nm == 0:
        return float('nan')

    u_count = 0
    for ms in m_scores:
        u_count += np.sum(ms > nm_scores) + 0.5 * np.sum(ms == nm_scores)
    return u_count / (n_m * n_nm)



## Pre-Registered Fair Comparison (Min-K%++ only)

This section keeps a single fair protocol to test how `standard` and `attnres`
compare under the **same Min-K%++ detector**.

Protocol:
- Models: `standard`, `attnres`
- Data: TinyStories train/val split from the loaded experiment
- Metric: `Min-K%++` at `k=10/20/40`
- Reporting: all k, both models, same sampled batches, no cherry-picking one best point
- Scope: smoke-test only (not benchmark)


In [ ]:

# =============================================================================
# Cell 8. Fair Min-K%++ sweep
# =============================================================================
K_VALUES_FAIR = [10, 20, 40]
N_MEMBER_FAIR = 64
N_NONMEMBER_FAIR = 64

DATA_PROTOCOLS = {
    'tinystories': DataConfig(
        dataset_type='tinystories',
        hf_dataset_name=experiment.data.hf_dataset_name,
        tokenizer_name=experiment.data.tokenizer_name,
        text_field=experiment.data.text_field,
        train_texts=experiment.data.train_texts if experiment.data.train_texts is not None else 4096,
        val_texts=experiment.data.val_texts if experiment.data.val_texts is not None else 1024,
        block_stride=max(1, mc.block_size),
        use_token_cache=experiment.data.use_token_cache,
        token_cache_dir=experiment.data.token_cache_dir,
    ),
}

def _take_n_batches(loader, n):
    inputs_parts = []
    targets_parts = []
    taken = 0
    for inputs, targets in loader:
        inputs_parts.append(inputs)
        targets_parts.append(targets)
        taken += inputs.size(0)
        if taken >= n:
            break
    if not inputs_parts:
        raise RuntimeError('TinyStories loader returned no batches.')
    all_inputs = torch.cat(inputs_parts, dim=0)[:n]
    all_targets = torch.cat(targets_parts, dim=0)[:n]
    if all_inputs.size(0) < n:
        raise RuntimeError(f'Need {n} samples, but only got {all_inputs.size(0)} from TinyStories loader.')
    return all_inputs, all_targets

def build_member_nonmember_for_data(data_cfg, model_cfg, seed, n_member, n_nonmember):
    sampling_model_cfg = deepcopy(model_cfg)
    train_loader, val_loader = build_dataloaders(
        model_config=sampling_model_cfg,
        data_config=data_cfg,
        batch_size=max(16, min(n_member, n_nonmember)),
        num_workers=0,
        seed=seed,
        distributed=False,
        verbose=True,
    )
    member_in, member_tg = _take_n_batches(train_loader, n_member)
    nonmember_in, nonmember_tg = _take_n_batches(val_loader, n_nonmember)
    return member_in, member_tg, nonmember_in, nonmember_tg

fair_rows = []
fair_scores = {}

for data_name, data_cfg in DATA_PROTOCOLS.items():
    member_in, member_tg, nonmember_in, nonmember_tg = build_member_nonmember_for_data(
        data_cfg=data_cfg,
        model_cfg=mc,
        seed=experiment.train.seed,
        n_member=N_MEMBER_FAIR,
        n_nonmember=N_NONMEMBER_FAIR,
    )

    for model_name in MODEL_VARIANTS:
        model = models[model_name]
        mem_lp, mem_mu, mem_sigma = compute_per_token_logprobs_and_stats(member_in, member_tg)
        nm_lp, nm_mu, nm_sigma = compute_per_token_logprobs_and_stats(nonmember_in, nonmember_tg)

        score_map = {}
        for k in K_VALUES_FAIR:
            score_map[f'MinKPP_{k}'] = {
                'member': min_k_pp_score(mem_lp, mem_mu, mem_sigma, k).numpy(),
                'nonmember': min_k_pp_score(nm_lp, nm_mu, nm_sigma, k).numpy(),
            }

        fair_scores[(data_name, model_name)] = score_map

        for metric_name, payload in score_map.items():
            auc = compute_auroc_manual(payload['member'], payload['nonmember'])
            fair_rows.append({
                'model': model_name,
                'data': data_name,
                'metric': 'MinKPP',
                'k': int(metric_name.split('_')[1]),
                'auc': float(auc),
            })

for row in fair_rows:
    other = 'attnres' if row['model'] == 'standard' else 'standard'
    competitor_auc = None
    for r in fair_rows:
        if r['model'] == other and r['data'] == row['data'] and r['metric'] == row['metric'] and r['k'] == row['k']:
            competitor_auc = r['auc']
            break
    row['above_random'] = bool(row['auc'] > 0.5)
    row['above_other_model'] = (None if competitor_auc is None else bool(row['auc'] > competitor_auc))

try:
    import pandas as pd
    fair_df = pd.DataFrame(fair_rows).sort_values(['data', 'k', 'model']).reset_index(drop=True)
    print('Fair Min-K%++ comparison table (all k, both models, all configured datasets):')
    print(fair_df.to_string(index=False))
except Exception:
    fair_df = None
    print('Fair Min-K%++ comparison rows:')
    for row in fair_rows:
        print(row)

save_json_artifact(fair_rows, 'minkpp_fair_rows.json')
if fair_df is not None:
    save_dataframe_artifact(fair_df, 'minkpp_fair_comparison.csv')


In [ ]:

# =============================================================================
# Cell 9. Fair combined visualization (Min-K%++ only)
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
fig.suptitle('Fair Standard vs AttnRes (Min-K%++ only; TinyStories sampling)', fontsize=14)

# Panel A: AUC vs k (all model/data curves)
ax = axes[0, 0]
for data_idx, data_name in enumerate(DATA_PROTOCOLS):
    for model_name in MODEL_VARIANTS:
        y = []
        for k in K_VALUES_FAIR:
            row = next(
                r for r in fair_rows
                if r['data'] == data_name and r['model'] == model_name and r['metric'] == 'MinKPP' and r['k'] == k
            )
            y.append(row['auc'])
        style = ['-', '--', '-.', ':'][data_idx % 4]
        ax.plot(K_VALUES_FAIR, y, marker='o', linestyle=style, linewidth=2, label=f'{model_name} | {data_name}')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.set_title('AUC vs k (Min-K%++)')
ax.set_xlabel('k (%)')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.0, 1.0)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel B: Delta (attnres - standard) across k
ax = axes[0, 1]
for data_idx, data_name in enumerate(DATA_PROTOCOLS):
    deltas = []
    for k in K_VALUES_FAIR:
        a = next(r['auc'] for r in fair_rows if r['data'] == data_name and r['model'] == 'attnres' and r['metric'] == 'MinKPP' and r['k'] == k)
        s = next(r['auc'] for r in fair_rows if r['data'] == data_name and r['model'] == 'standard' and r['metric'] == 'MinKPP' and r['k'] == k)
        deltas.append(float(a - s))
    style = ['-', '--', '-.', ':'][data_idx % 4]
    ax.plot(K_VALUES_FAIR, deltas, marker='o', linestyle=style, linewidth=2, label=f'{data_name}: attnres-standard')
ax.axhline(0.0, color='gray', linestyle=':', linewidth=1)
ax.set_title('AUC delta vs k (Min-K%++, attnres - standard)')
ax.set_xlabel('k (%)')
ax.set_ylabel('Delta AUC')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel C: Representative score distribution (k=20)
ax = axes[1, 0]
rep_data = next(iter(DATA_PROTOCOLS.keys()))
rep_metric_key = 'MinKPP_20'
for model_name, color in [('standard', 'tab:blue'), ('attnres', 'tab:orange')]:
    m = fair_scores[(rep_data, model_name)][rep_metric_key]['member']
    nm = fair_scores[(rep_data, model_name)][rep_metric_key]['nonmember']
    ax.hist(m, bins=24, histtype='step', linewidth=2, color=color, label=f'{model_name} member')
    ax.hist(nm, bins=24, histtype='step', linewidth=2, linestyle='--', color=color, label=f'{model_name} non-member')
ax.set_title(f'Representative distribution ({rep_data}, Min-K%++, k=20)')
ax.set_xlabel('score')
ax.set_ylabel('count')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel D: Mean AUC by k
ax = axes[1, 1]
x = np.arange(len(K_VALUES_FAIR))
w = 0.35
vals_std = []
vals_att = []
for k in K_VALUES_FAIR:
    auc_std = [r['auc'] for r in fair_rows if r['model'] == 'standard' and r['metric'] == 'MinKPP' and r['k'] == k]
    auc_att = [r['auc'] for r in fair_rows if r['model'] == 'attnres' and r['metric'] == 'MinKPP' and r['k'] == k]
    vals_std.append(float(np.mean(auc_std)))
    vals_att.append(float(np.mean(auc_att)))
ax.bar(x - w / 2, vals_std, width=w, label='standard')
ax.bar(x + w / 2, vals_att, width=w, label='attnres')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f'MinKPP@{k}' for k in K_VALUES_FAIR])
ax.set_ylim(0.0, 1.0)
ax.set_title('Mean AUC summary (Min-K%++ only)')
ax.set_ylabel('AUC-ROC')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
save_figure_artifact('minkpp_fair_combined_visualization.png', fig)
plt.show()



## Focus of this notebook: Min-K%++ and cross-k stability

This notebook now keeps **Min-K%++ only** (`k=10/20/40`, `standard+attnres`, shared TinyStories sampling).

Re-organization for this iteration:
- **Part A:** full Min-K%++ results for both models.
- **Part B:** dedicated model-vs-model Min-K%++ summary across k.
- **Part C:** Min-K%++ cross-k stability diagnostics.

This notebook is now a single-metric Min-K%++ workflow.


In [ ]:

# =============================================================================
# Cell 11. Min-K%++ summary (standard vs attnres)
# =============================================================================

def _lookup_auc(rows, *, data, model, metric, k):
    for r in rows:
        if r['data'] == data and r['model'] == model and r['metric'] == metric and r['k'] == k:
            return float(r['auc'])
    raise KeyError(f'missing row: data={data}, model={model}, metric={metric}, k={k}')

DATASETS_ORDER = list(DATA_PROTOCOLS.keys())
minkpp_summary_rows = []

for k in K_VALUES_FAIR:
    std_by_data = [_lookup_auc(fair_rows, data=d, model='standard', metric='MinKPP', k=k) for d in DATASETS_ORDER]
    att_by_data = [_lookup_auc(fair_rows, data=d, model='attnres', metric='MinKPP', k=k) for d in DATASETS_ORDER]

    std_auc = float(np.mean(std_by_data))
    att_auc = float(np.mean(att_by_data))
    delta_auc = float(att_auc - std_auc)
    leader = 'attnres' if delta_auc > 0 else 'standard' if delta_auc < 0 else 'tie'

    minkpp_summary_rows.append({
        'k': int(k),
        'standard_auc': std_auc,
        'attnres_auc': att_auc,
        'delta_auc': delta_auc,
        'leader': leader,
    })

try:
    import pandas as pd
    minkpp_summary_df = pd.DataFrame(minkpp_summary_rows)
    minkpp_summary_df = minkpp_summary_df[['k', 'standard_auc', 'attnres_auc', 'delta_auc', 'leader']]
    print('Min-K%++ comparison table (AUC averaged over configured datasets):')
    print(minkpp_summary_df.to_string(index=False))
except Exception:
    minkpp_summary_df = None
    print('Min-K%++ comparison table (fallback print):')
    for row in minkpp_summary_rows:
        print(row)

leaders = [r['leader'] for r in minkpp_summary_rows if r['leader'] != 'tie']
max_gap_row = max(minkpp_summary_rows, key=lambda x: abs(x['delta_auc']))
minkpp_claim = {
    'all_k_same_leader': bool(len(set(leaders)) == 1) if leaders else True,
    'dominant_model_if_any': leaders[0] if leaders and len(set(leaders)) == 1 else 'mixed',
    'largest_gap_k': int(max_gap_row['k']),
    'largest_gap_delta_auc': float(max_gap_row['delta_auc']),
    'largest_gap_leader': max_gap_row['leader'],
}

print()
print('Min-K%++ conclusion (auto):')
print(f"- same leader on all k: {minkpp_claim['all_k_same_leader']}")
print(f"- dominant model across k: {minkpp_claim['dominant_model_if_any']}")
print(f"- largest gap at k={minkpp_claim['largest_gap_k']} with delta={minkpp_claim['largest_gap_delta_auc']:+.4f}")
print(f"- leader at the largest gap: {minkpp_claim['largest_gap_leader']}")

save_json_artifact({'rows': minkpp_summary_rows, 'claim': minkpp_claim}, 'minkpp_summary.json')
if minkpp_summary_df is not None:
    save_dataframe_artifact(minkpp_summary_df, 'minkpp_summary.csv')


In [ ]:

# =============================================================================
# Cell 12. Min-K%++ line plot
# =============================================================================
k_vals = [r['k'] for r in minkpp_summary_rows]
std_vals = [r['standard_auc'] for r in minkpp_summary_rows]
att_vals = [r['attnres_auc'] for r in minkpp_summary_rows]
delta_vals = [r['delta_auc'] for r in minkpp_summary_rows]

plt.figure(figsize=(8, 5))
plt.plot(k_vals, std_vals, marker='o', linewidth=2, label='standard')
plt.plot(k_vals, att_vals, marker='o', linewidth=2, label='attnres')
for k, d in zip(k_vals, delta_vals):
    plt.text(k, max(std_vals + att_vals) + 0.005, f'Δ={d:+.3f}', ha='center', va='bottom', fontsize=9)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.ylim(0.0, 1.0)
plt.xlabel('k (%)')
plt.ylabel('AUC-ROC')
plt.title('Min-K%++ comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
save_figure_artifact('minkpp_comparison.png')
plt.show()


In [ ]:

# =============================================================================
# Cell 13. Cross-k stability summary (Min-K%++ only)
# =============================================================================

def compute_cross_k_stability(rows, metric_type):
    out = []
    for model_name in MODEL_VARIANTS:
        auc_by_k = []
        for k in K_VALUES_FAIR:
            vals = [
                float(r['auc'])
                for r in rows
                if r['model'] == model_name and r['metric'] == metric_type and r['k'] == k
            ]
            auc_by_k.append(float(np.mean(vals)))

        range_across_k = float(max(auc_by_k) - min(auc_by_k))
        std_across_k = float(np.std(auc_by_k))
        drop_10_to_40 = float(abs(auc_by_k[0] - auc_by_k[-1]))
        peak_idx = int(np.argmax(auc_by_k))
        peak_auc = float(auc_by_k[peak_idx])
        peak_k = int(K_VALUES_FAIR[peak_idx])

        out.append({
            'model': model_name,
            'metric_type': 'Min-K++',
            'range_across_k': range_across_k,
            'std_across_k': std_across_k,
            'drop_10_to_40': drop_10_to_40,
            'peak_auc': peak_auc,
            'peak_k': peak_k,
        })
    return out

stability_rows = compute_cross_k_stability(fair_rows, 'MinKPP')

try:
    import pandas as pd
    stability_df = pd.DataFrame(stability_rows)
    stability_df = stability_df[['model', 'metric_type', 'range_across_k', 'std_across_k', 'drop_10_to_40', 'peak_auc', 'peak_k']]
    print('Min-K%++ cross-k stability summary table:')
    print(stability_df.to_string(index=False))
except Exception:
    stability_df = None
    print('Min-K%++ cross-k stability summary table (fallback print):')
    for row in stability_rows:
        print(row)

more_stable = min(stability_rows, key=lambda r: (r['std_across_k'], r['range_across_k'], r['drop_10_to_40']))['model']
higher_peak = max(stability_rows, key=lambda r: r['peak_auc'])['model']
print()
print(f"[Min-K%++] more stable across k: {more_stable}")
print(f"[Min-K%++] higher peak AUC across k: {higher_peak}")

save_json_artifact(stability_rows, 'minkpp_cross_k_stability.json')
if stability_df is not None:
    save_dataframe_artifact(stability_df, 'minkpp_cross_k_stability.csv')


In [ ]:

# =============================================================================
# Cell 14. Cross-k stability visualization (Min-K%++ only)
# =============================================================================

def _mkpp_stat(model_name, key):
    row = next(r for r in stability_rows if r['model'] == model_name)
    return float(row[key])

stats = ['range_across_k', 'std_across_k', 'drop_10_to_40', 'peak_auc']
labels = ['range across k (low=stable)', 'std across k (low=stable)', 'drop 10->40 (low=stable)', 'peak AUC (high=better)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Cross-k stability & peak (Min-K%++ only, standard vs attnres)', fontsize=13)

for ax, key, title in zip(axes.flatten(), stats, labels):
    vals = [_mkpp_stat('standard', key), _mkpp_stat('attnres', key)]
    ax.bar(['standard', 'attnres'], vals, color=['tab:blue', 'tab:orange'])
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.02, 1, 0.95])
save_figure_artifact('minkpp_cross_k_stability.png', fig)
plt.show()



## Final interpretation (Min-K%++ only, smoke-test scope)

- Keep the claim tied to `Min-K%++`; this notebook is now a single-metric comparison.
- Compare `standard` and `attnres` jointly across all configured `k` values instead of cherry-picking one point.
- Use the summary table for average AUC gaps and the stability table for robustness across `k`.
- These are smoke-test observations, not benchmark-level proof.



## Concise takeaway (smoke-test scope)

- This notebook now reports `Min-K%++` only.
- Judge model preference from the combined table, the k-wise line plot, and the Min-K%++ stability section together.
- Treat the result as a smoke-test signal, not a final benchmark claim.
